# Dự Đoán Giá Bitcoin Sử Dụng Hồi Quy và RNN

## Tổng quan dự án
Notebook này minh họa quy trình hoàn chỉnh từ đầu đến cuối để dự đoán giá Bitcoin sử dụng:
- **Hồi quy Tuyến tính (Linear Regression)** (Mô hình cơ bản)
- **RNN Đơn giản (Simple RNN)** - Mạng nơ-ron hồi quy
- **LSTM** - Mạng nhớ ngắn hạn dài hạn

### Mục tiêu
1. Thu thập dữ liệu giá lịch sử Bitcoin
2. Kỹ thuật hóa chỉ số kỹ thuật thành đặc trưng
3. Tiền xử lý dữ liệu cho machine learning
4. Huấn luyện và đánh giá nhiều mô hình
5. So sánh hiệu suất mô hình
6. Trực quan hóa kết quả

### Tác giả
Dự án Dự đoán Giá Bitcoin - 2026

## Mục lục
1. [Thiết lập và nhập thư viện](#1-thiết-lập-và-nhập-thư-viện)
2. [Thu thập dữ liệu](#2-thu-thập-dữ-liệu)
3. [Kỹ thuật đặc trưng](#3-kỹ-thuật-đặc-trưng)
4. [Tiền xử lý dữ liệu](#4-tiền-xử-lý-dữ-liệu)
5. [Mô hình Hồi quy Tuyến tính](#5-mô-hình-hồi-quy-tuyến-tính)
6. [Mô hình RNN](#6-mô-hình-rnn)
7. [Mô hình LSTM](#7-mô-hình-lstm)
8. [So sánh mô hình](#8-so-sánh-mô-hình)
9. [Trực quan hóa](#9-trực-quán-hóa)
10. [Kết luận](#10-kết-luận)

## 1. Thiết lập và nhập thư viện

In [1]:
# Nhập các thư viện cần thiết
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Thêm thư mục src vào đường dẫn
sys.path.append('../src')

# Nhập các module dự án
from data_collection import BitcoinDataCollector
from feature_engineering import TechnicalIndicators
from data_preprocessing import DataPreprocessor
from linear_regression_model import LinearRegressionModel
from rnn_model import RNNModel
from lstm_model import LSTMModel
from evaluation import ModelEvaluator
from visualization import BitcoinVisualizer

# Thiết lập kiểu cho trực quan hóa
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Tất cả các thư viện đã được nhập thành công!")

ModuleNotFoundError: No module named 'seaborn'

## 2. Thu thập dữ liệu

Chúng ta sẽ lấy 5 năm dữ liệu giá Bitcoin hàng ngày từ Yahoo Finance sử dụng thư viện `yfinance`.

In [ ]:
# Khởi tạo bộ thu thập dữ liệu
collector = BitcoinDataCollector(ticker="BTC-USD")

# Lấy 5 năm dữ liệu hàng ngày
print("Đang lấy dữ liệu Bitcoin...")
data = collector.fetch_data(period="5y", interval="1d")

# Hiển thị vài hàng đầu tiên
print("\n5 hàng đầu tiên của dữ liệu:")
print(data.head())

# Hiển thị tóm tắt dữ liệu
print("\nTóm tắt dữ liệu:")
summary = collector.get_data_summary()
for key, value in summary.items():
    print(f"{key}: {value}")

# Lưu dữ liệu thô
collector.save_to_csv("../data/bitcoin_data.csv")
print("\nDữ liệu đã được lưu vào ../data/bitcoin_data.csv")

### Hiểu về dữ liệu

Dữ liệu chứa thông tin OHLCV (Open, High, Low, Close, Volume):
- **Open**: Giá tại thời điểm mở cửa ngày
- **High**: Giá cao nhất trong ngày
- **Low**: Giá thấp nhất trong ngày
- **Close**: Giá tại thời điểm đóng cửa ngày (quan trọng nhất)
- **Volume**: Khối lượng giao dịch

In [ ]:
# Hiển thị thống kê cơ bản
print("Thống kê dữ liệu:")
print(data.describe())

# Kiểm tra giá trị thiếu
print("\nGiá trị thiếu:")
print(data.isnull().sum())

## 3. Kỹ thuật đặc trưng

Chúng ta sẽ tạo các chỉ số kỹ thuật để giúp các mô hình đưa ra dự đoán tốt hơn.

In [ ]:
# Khởi tạo chỉ số kỹ thuật
ti = TechnicalIndicators(data)

# Thêm tất cả chỉ số kỹ thuật
print("Đang thêm chỉ số kỹ thuật...")
data_with_features = ti.add_all_indicators()

# Hiển thị các cột đặc trưng
print("\nCác cột đặc trưng đã thêm:")
feature_columns = ti.get_feature_columns()
print(feature_columns)

# Hiển thị mẫu dữ liệu với đặc trưng
print("\nMẫu dữ liệu với đặc trưng (5 hàng cuối):")
print(data_with_features.tail())

# Lưu dữ liệu với đặc trưng
data_with_features.to_csv("../data/bitcoin_data_with_features.csv", index=False)
print("\nDữ liệu với đặc trưng đã được lưu vào ../data/bitcoin_data_with_features.csv")

### Giải thích chỉ số kỹ thuật

1. **MA20 & MA50**: Đường trung bình - Làm mịn dữ liệu giá để xác định xu hướng
2. **RSI**: Chỉ số sức mạnh tương đối - Đo động lượng (mua quá > 70, bán quá < 30)
3. **MACD**: Hội tụ phân kỳ đường trung bình - Hiển thị sức mạnh và hướng của xu hướng
4. **ATR**: Phạm vi thực tế trung bình - Đo độ biến động
5. **Thay đổi giá**: Phần trăm thay đổi qua các giai đoạn khác nhau - Nắm bắt động lượng
6. **Đặc trưng khối lượng**: Chỉ số khối lượng - Hiển thị sức mạnh giao dịch

## 4. Tiền xử lý dữ liệu

Chúng ta sẽ tiền xử lý dữ liệu cho machine learning:
- Xử lý giá trị thiếu
- Chuẩn hóa dữ liệu sử dụng MinMaxScaler
- Tạo chuỗi cho mô hình RNN/LSTM
- Chia thành tập train/test

In [ ]:
# Khởi tạo bộ tiền xử lý
preprocessor = DataPreprocessor(data_with_features)

# Xử lý giá trị thiếu
print("Đang xử lý giá trị thiếu...")
data_processed = preprocessor.handle_missing_values(method='forward_fill')

# Sắp xếp theo ngày
print("Đang sắp xếp dữ liệu theo ngày...")
data_processed = preprocessor.sort_by_date()

# Chuẩn hóa dữ liệu
print("Đang chuẩn hóa dữ liệu...")
feature_cols = [col for col in data_processed.columns if col != 'Date']
data_normalized = preprocessor.normalize_data(columns_to_scale=feature_cols)

# Định nghĩa đặc trưng cho các mô hình
selected_features = ['Close', 'MA20', 'MA50', 'RSI', 'MACD', 'ATR']
print(f"\nCác đặc trưng đã chọn: {selected_features}")

In [ ]:
# Tạo chuỗi cho RNN/LSTM (30 ngày để dự đoán ngày tiếp theo)
print("Đang tạo chuỗi cho các mô hình RNN/LSTM...")
X, y = preprocessor.create_sequences(
    feature_columns=selected_features,
    target_column='Close',
    sequence_length=30
)

print(f"X shape: {X.shape} (samples, sequence_length, features)")
print(f"y shape: {y.shape} (samples,)")

# Chia thành train/test (80% train, 20% test)
X_train, X_test, y_train, y_test = preprocessor.train_test_split(X, y, test_size=0.2)

print(f"\nSố lượng mẫu huấn luyện: {len(X_train)}")
print(f"Số lượng mẫu kiểm tra: {len(X_test)}")

# Chia thêm tập huấn luyện thành train/validation cho mạng nơ-ron
val_size = int(len(X_train) * 0.2)
X_val = X_train[-val_size:]
y_val = y_train[-val_size:]
X_train_nn = X_train[:-val_size]
y_train_nn = y_train[:-val_size]

print(f"\nSố lượng mẫu huấn luyện Neural Network: {len(X_train_nn)}")
print(f"Số lượng mẫu xác thực Neural Network: {len(X_val)}")

# Lưu dữ liệu đã tiền xử lý
np.save('../data/X_train.npy', X_train)
np.save('../data/X_test.npy', X_test)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)
print("\nDữ liệu đã tiền xử lý được lưu vào thư mục ../data/")

### Tại sao độ dài chuỗi là 30?

Chúng ta sử dụng 30 ngày dữ liệu lịch sử để dự đoán giá ngày tiếp theo vì:
- 30 ngày ≈ 1 tháng dữ liệu giao dịch
- Nắm bắt xu hướng và mẫu ngắn hạn
- Thực tế phổ biến trong chuỗi thời gian tài chính
- Cân bằng giữa ngữ cảnh và hiệu quả tính toán

## 5. Mô hình Hồi quy Tuyến tính

### Nền tảng toán học

**Công thức Hồi quy Tuyến tính:**
```
y = w·x + b
```

Trong đó:
- y = giá dự đoán
- w = trọng số (hệ số)
- x = đặc trưng đầu vào
- b = độ lệch (intercept)

**Mục tiêu huấn luyện:** Tối thiểu hóa Sai số Bình phương Trung bình (MSE)
```
MSE = (1/n) × Σ(yᵢ - ŷᵢ)²
```

In [ ]:
# Đối với Hồi quy Tuyến tính, chúng ta sử dụng ngày cuối cùng của mỗi chuỗi (không phải toàn bộ chuỗi)
X_train_lr = X_train[:, -1, :]  # Lấy ngày cuối cùng của mỗi chuỗi
X_test_lr = X_test[:, -1, :]

print(f"Đầu vào Hồi quy Tuyến tính Shapes:")
print(f"  X_train: {X_train_lr.shape}")
print(f"  X_test: {X_test_lr.shape}")

# Khởi tạo và huấn luyện mô hình Hồi quy Tuyến tính
print("\nĐang huấn luyện mô hình Hồi quy Tuyến tính...")
lr_model = LinearRegressionModel()
lr_model.train(X_train_lr, y_train)

# Đánh giá mô hình
print("\nĐang đánh giá mô hình Hồi quy Tuyến tính...")
lr_metrics = lr_model.evaluate(X_test_lr, y_test)

# Tạo dự đoán
y_pred_lr = lr_model.predict(X_test_lr)

# Lưu dự đoán
np.save('../results/lr_predictions.npy', y_pred_lr)
print("\nDự đoán đã được lưu vào ../results/lr_predictions.npy")

# Tầm quan trọng đặc trưng
print("\nTầm quan trọng đặc trưng:")
importance_df = lr_model.get_feature_importance(selected_features)
print(importance_df)

### Giải thích kết quả Hồi quy Tuyến tính

- **Hệ số**: Hiển thị tác động của từng đặc trưng lên dự đoán
- **Điểm R²**: Tỷ lệ phương sai được giải thích (0 đến 1)
- **MSE/RMSE/MAE**: Các chỉ số lỗi (càng thấp càng tốt)

## 6. Mô hình RNN

### Nền tảng toán học

**Công thức trạng thái ẩn RNN:**
```
hₜ = f(W·xₜ + U·hₜ₋₁ + b)
```

Trong đó:
- hₜ = trạng thái ẩn tại thời điểm t
- xₜ = đầu vào tại thời điểm t
- hₜ₋₁ = trạng thái ẩn tại thời điểm t-1
- W = ma trận trọng số đầu vào
- U = ma trận trọng số hồi quy
- b = độ lệch
- f = hàm kích hoạt (thường là tanh)

### Tại sao RNN cho chuỗi thời gian?
- **Bộ nhớ**: Trạng thái ẩn duy trì thông tin từ các bước thời gian trước
- **Tuần tự**: Xử lý dữ liệu theo thứ tự, tôn trọng phụ thuộc thời gian
- Linh hoạt: Có thể xử lý chuỗi độ dài biến thiên

In [ ]:
# Khởi tạo mô hình RNN
print("Đang khởi tạo mô hình RNN...")
rnn_model = RNNModel(
    sequence_length=X_train_nn.shape[1],
    n_features=X_train_nn.shape[2],
    rnn_units=50,
    dropout_rate=0.2
)

# Xây dựng mô hình
rnn_model.build_model()

# Huấn luyện mô hình
print("\nĐang huấn luyện mô hình RNN...")
history_rnn = rnn_model.train(
    X_train_nn, y_train_nn,
    X_val, y_val,
    epochs=50,
    batch_size=32,
    patience=10
)

# Đánh giá mô hình
print("\nĐang đánh giá mô hình RNN...")
rnn_metrics = rnn_model.evaluate(X_test, y_test)

# Tạo dự đoán
y_pred_rnn = rnn_model.predict(X_test)

# Lưu mô hình và dự đoán
rnn_model.save_model()
rnn_model.save_training_history()
np.save('../results/rnn_predictions.npy', y_pred_rnn)
print("\nMô hình và dự đoán đã được lưu")

### Giải thích kiến trúc RNN

**Các lớp:**
1. **SimpleRNN(50)**: 50 đơn vị hồi quy xử lý chuỗi
2. **Dropout(0.2)**: Ngẫu nhiên loại bỏ 20% nơ-ron để ngăn chặn overfitting
3. **Dense(1)**: Lớp đầu ra dự đoán giá ngày tiếp theo

**Huấn luyện:**
- Optimizer: Adam (tốc độ học thích ứng)
- Loss: MSE (Sai số bình phương trung bình)
- Early stopping: Dừng nếu loss xác thực không cải thiện

## 7. Mô hình LSTM

### Nền tảng toán học

**Công thức ô nhớ LSTM:**
```
fₜ = sigmoid(Wf·[hₜ₋₁, xₜ] + bf)  # Cổng quên (forget gate)
iₜ = sigmoid(Wi·[hₜ₋₁, xₜ] + bi)  # Cổng đầu vào (input gate)
C̃ₜ = tanh(WC·[hₜ₋₁, xₜ] + bC)    # Trạng thái ô nhớ ứng viên
Cₜ = fₜ·Cₜ₋₁ + iₜ·C̃ₜ            # Trạng thái ô nhớ
oₜ = sigmoid(Wo·[hₜ₋₁, xₜ] + bo)  # Cổng đầu ra (output gate)
hₜ = oₜ·tanh(Cₜ)                   # Trạng thái ẩn
```

### Tại sao LSTM thay vì RNN Đơn giản?
- **Bộ nhớ dài hạn**: Trạng thái ô nhớ duy trì thông tin qua các giai đoạn dài
- **Không có gradient biến mất**: Các cổng kiểm soát luồng gradient
- **Tốt hơn cho chuỗi dài**: Có thể học phụ thuộc qua nhiều tuần/tháng

In [ ]:
# Khởi tạo mô hình LSTM
print("Đang khởi tạo mô hình LSTM...")
lstm_model = LSTMModel(
    sequence_length=X_train_nn.shape[1],
    n_features=X_train_nn.shape[2],
    lstm_units=50,
    dropout_rate=0.2
)

# Xây dựng mô hình
lstm_model.build_model()

# Huấn luyện mô hình
print("\nĐang huấn luyện mô hình LSTM...")
history_lstm = lstm_model.train(
    X_train_nn, y_train_nn,
    X_val, y_val,
    epochs=50,
    batch_size=32,
    patience=10
)

# Đánh giá mô hình
print("\nĐang đánh giá mô hình LSTM...")
lstm_metrics = lstm_model.evaluate(X_test, y_test)

# Tạo dự đoán
y_pred_lstm = lstm_model.predict(X_test)

# Lưu mô hình và dự đoán
lstm_model.save_model()
lstm_model.save_training_history()
np.save('../results/lstm_predictions.npy', y_pred_lstm)
print("\nMô hình và dự đoán đã được lưu")

### Giải thích kiến trúc LSTM

**Các lớp:**
1. **LSTM(50)**: 50 đơn vị LSTM với ô nhớ
2. **Dropout(0.2)**: Điều chuẩn để ngăn chặn overfitting
3. **Dense(1)**: Lớp đầu ra cho dự đoán giá

**Các cổng LSTM:**
- **Cổng quên (forget gate)**: Quyết định loại bỏ gì từ trạng thái ô nhớ
- **Cổng đầu vào (input gate)**: Quyết định lưu gì vào trạng thái ô nhớ
- **Cổng đầu ra (output gate)**: Quyết định xuất gì ra trạng thái ẩn

## 8. So sánh mô hình

### Giải thích chỉ số đánh giá

**MSE (Mean Squared Error - Sai số bình phương trung bình):**
```
MSE = (1/n) × Σ(yᵢ - ŷᵢ)²
```
- Phạt nặng các sai số lớn
- Đơn vị: bình phương (ví dụ: USD²)

**RMSE (Root Mean Squared Error - Căn bậc hai sai số bình phương trung bình):**
```
RMSE = √MSE
```
- Cùng đơn vị với mục tiêu (ví dụ: USD)
- Dễ hiểu hơn MSE

**MAE (Mean Absolute Error - Sai số tuyệt đối trung bình):**
```
MAE = (1/n) × Σ|yᵢ - ŷᵢ|
```
- Ít nhạy cảm với giá trị ngoại lai
- Dễ hiểu hơn

**R² (Hệ số xác định - R-squared):**
```
R² = 1 - (Σ(yᵢ - ŷᵢ)² / Σ(yᵢ - ȳ)²)
```
- Tỷ lệ phương sai được giải thích
- Phạm vi: -∞ đến 1 (càng cao càng tốt)

In [ ]:
# Khởi tạo bộ đánh giá
evaluator = ModelEvaluator()

# Đánh giá tất cả mô hình
print("Đang đánh giá tất cả các mô hình...")
print("\n" + "="*60)
print("Hồi quy Tuyến tính")
print("="*60)
metrics_lr = evaluator.evaluate_model(y_test, y_pred_lr, "Linear Regression")

print("\n" + "="*60)
print("RNN")
print("="*60)
metrics_rnn = evaluator.evaluate_model(y_test, y_pred_rnn, "RNN")

print("\n" + "="*60)
print("LSTM")
print("="*60)
metrics_lstm = evaluator.evaluate_model(y_test, y_pred_lstm, "LSTM")

# Tạo bảng so sánh
print("\n" + "="*60)
print("SO SÁNH MÔ HÌNH")
print("="*60)
comparison_df = evaluator.create_comparison_table()

# Lưu bảng so sánh
evaluator.save_comparison_table(comparison_df)
print("\nBảng so sánh đã được lưu vào ../results/model_comparison.csv")

# Lấy mô hình tốt nhất
best_model = evaluator.get_best_model(metric='RMSE')
print(f"\nMô hình tốt nhất theo RMSE: {best_model[0]}")

## 9. Trực quan hóa

Hãy tạo các biểu đồ trực quan để hiểu rõ hơn về kết quả của chúng ta.

In [ ]:
# Khởi tạo bộ trực quan hóa
visualizer = BitcoinVisualizer(save_dir='../results')

# Vẽ lịch sử giá
print("Đang tạo trực quan hóa...")
visualizer.plot_price_history(data, title='Lịch sử giá Bitcoin (5 năm)')
print("✓ Biểu đồ lịch sử giá đã được lưu")

In [ ]:
# Vẽ chỉ số kỹ thuật
visualizer.plot_technical_indicators(data_with_features, 
                                      indicators=['MA20', 'MA50', 'RSI', 'MACD'])
print("✓ Biểu đồ chỉ số kỹ thuật đã được lưu")

In [ ]:
# Vẽ dự đoán cho từng mô hình
visualizer.plot_predictions(y_test, y_pred_lr, "Linear Regression")
print("✓ Biểu đồ dự đoán Hồi quy Tuyến tính đã được lưu")

visualizer.plot_predictions(y_test, y_pred_rnn, "RNN")
print("✓ Biểu đồ dự đoán RNN đã được lưu")

visualizer.plot_predictions(y_test, y_pred_lstm, "LSTM")
print("✓ Biểu đồ dự đoán LSTM đã được lưu")

In [ ]:
# Vẽ đường cong mất mát cho mạng nơ-ron
visualizer.plot_loss_curve('../results/rnn_training_history.json', 'RNN')
print("✓ Đường cong mất mát RNN đã được lưu")

visualizer.plot_loss_curve('../results/lstm_training_history.json', 'LSTM')
print("✓ Đường cong mất mát LSTM đã được lưu")

In [ ]:
# Vẽ so sánh mô hình
visualizer.plot_model_comparison(comparison_df)
print("✓ Biểu đồ so sánh mô hình đã được lưu")

In [ ]:
# Vẽ phần dư cho từng mô hình
visualizer.plot_residuals(y_test, y_pred_lr, "Linear Regression")
print("✓ Biểu đồ phần dư Hồi quy Tuyến tính đã được lưu")

visualizer.plot_residuals(y_test, y_pred_rnn, "RNN")
print("✓ Biểu đồ phần dư RNN đã được lưu")

visualizer.plot_residuals(y_test, y_pred_lstm, "LSTM")
print("✓ Biểu đồ phần dư LSTM đã được lưu")

In [ ]:
# Vẽ bản đồ nhiệt tương quan
visualizer.plot_correlation_heatmap(data_normalized, selected_features)
print("✓ Bản đồ nhiệt tương quan đã được lưu")

## 10. Kết luận

### Tóm tắt kết quả

Chúng ta đã xây dựng thành công và so sánh ba mô hình cho dự đoán giá Bitcoin:

1. **Hồi quy Tuyến tính** (Mô hình cơ bản)
   - Đơn giản và dễ hiểu
   - Huấn luyện nhanh
   - Tốt làm cơ sở so sánh

2. **RNN Đơn giản**
   - Nắm bắt phụ thuộc thời gian
   - Sử dụng thông tin chuỗi
   - Phức tạp hơn hồi quy tuyến tính

3. **LSTM**
   - RNN nâng cao với ô nhớ
   - Xử lý phụ thuộc dài hạn
   - Mô hình tinh vi nhất

### Phát hiện chính
- Mạng nơ-ron (RNN/LSTM) có thể nắm bắt các mẫu thời gian phức tạp
- Chỉ số kỹ thuật cung cấp đặc trưng hữu ích cho dự đoán
- Hiệu suất mô hình phụ thuộc vào chất lượng dữ liệu và kỹ thuật đặc trưng
- Không mô hình nào hoàn hảo - thị trường tài chính vốn dĩ khó dự đoán

### Hạn chế
- **Biến động thị trường**: Thị trường crypto rất biến động
- **Yếu tố bên ngoài**: Tin tức, quy định, tâm lý không được bao gồm
- **Không dừng (Non-stationarity)**: Điều kiện thị trường thay đổi theo thời gian
- **Rủi ro overfitting**: Mô hình phức tạp có thể overfit dữ liệu lịch sử

### Hướng phát triển

1. **Nguồn dữ liệu bổ sung**
   - Tâm lý mạng xã hội (Twitter/X)
   - Phân tích tin tức sử dụng NLP/LLM
   - Chỉ số on-chain
   - Chỉ số kinh tế vĩ mô (DXY, Vàng)

2. **Mô hình nâng cao**
   - BiLSTM (LSTM hai chiều)
   - GRU (Đơn vị hồi quy có cổng)
   - Cơ chế Transformer/Attention
   - Phương pháp ensemble

3. **Ứng dụng giao dịch**
   - Tạo tín hiệu giao dịch
   - Quản lý rủi ro
   - Tối ưu hóa danh mục đầu tư
   - Hệ thống dự đoán thời gian thực

4. **Mở rộng nghiên cứu**
   - Tinh chỉnh siêu tham số (Grid Search, Tối ưu hóa Bayesian)
   - Cross-validation cho chuỗi thời gian
   - Phân tích tầm quan trọng đặc trưng
   - AI có thể giải thích (giá trị SHAP)

### Tài liệu tham khảo
- Tài liệu yfinance
- Tài liệu TensorFlow/Keras
- Tài liệu scikit-learn
- Văn học phân tích kỹ thuật
- Deep Learning cho chuỗi thời gian (Goodfellow et al.)

---

**Kết thúc Notebook**

Notebook này cung cấp quy trình hoàn chỉnh từ đầu đến cuối cho dự đoán giá Bitcoin sử dụng các mô hình machine learning và deep learning. Code có tính mô-đun, được giải thích chi tiết, và có thể dễ dàng mở rộng cho nghiên cứu hoặc trường hợp sản xuất.